# Paper Figures

This notebook collects the code used to generate publication-ready figures for the thesis/paper.

## Figure 1
Overall Resolution Accuracy Across Oracle Configurations

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Polygon, Rectangle

output_dir = Path('figures')
output_dir.mkdir(exist_ok=True)


In [ ]:
# ── Global style ──────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8.5,
})

# ── Data ─────────────────────────────────────────
labels = [
    'Published\nKalshiBench Best',
    'GPT-5\nNano',
    'DeepSeek\nV3',
    'Llama\n3.3-70B',
    'Majority\nVote',
    'Confidence-\nWeighted',
    'Deliberation\n(2-round)'
]
accuracies = [69.3, 82.34, 82.42, 81.24, 82.93, 83.43, 76.11]
groups = [-1, 0, 0, 0, 1, 1, 2]  # -1=published prior result, 0=single models, 1=Arch A, 2=Arch B

# ── Colors ─────────────────────────────────────
palette = {
    -1: '#C9C3B8',  # warm stone gray
     0: '#6F86C3',  # muted periwinkle/slate blue
     1: '#5C8A72',  # sage green
     2: '#C47C5A',  # terracotta
}
bar_colors = [palette[g] for g in groups]

# ── Figure setup ─────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5.8))

# Bar positions with clearer separation between conceptual groups
positions = []
x = 0
for i, g in enumerate(groups):
    if i > 0 and groups[i] != groups[i - 1]:
        x += 0.9   # inter-group gap
    positions.append(x)
    x += 0.8      # intra-group spacing
positions = np.array(positions)

# ── Draw bars ───────────────────────────────────
bar_width = 0.58
best_bar_index = 5  # Confidence-weighted vote

def draw_bars(ax):
    for i, (pos, acc, color) in enumerate(zip(positions, accuracies, bar_colors)):
        if groups[i] == -1:
            ax.bar(
                pos, acc, width=bar_width,
                color=color, edgecolor='#8A847A',
                linewidth=1.0, hatch='//', alpha=0.9, zorder=3
            )
        else:
            edge = 'black' if i == best_bar_index else 'white'
            lw = 1.4 if i == best_bar_index else 0.8
            ax.bar(
                pos, acc, width=bar_width,
                color=color, edgecolor=edge,
                linewidth=lw, zorder=3
            )

draw_bars(ax)

# ── Value labels ──────────────────────────────────
for pos, acc in zip(positions, accuracies):
    ax.text(
        pos, acc + 0.7, f'{acc:.1f}%',
        ha='center', va='bottom',
        fontsize=9, color='#333333'
    )

# ── Axes ──────────────────────────────────────
ax.set_ylim(0, 100)
ax.set_ylabel('Accuracy (%)', labelpad=8)

ax.set_xticks(positions)
ax.set_xticklabels(labels, ha='center')

ax.yaxis.set_major_locator(mticker.MultipleLocator(10))
ax.yaxis.set_minor_locator(mticker.MultipleLocator(5))
ax.grid(axis='y', which='major', linewidth=0.5, color='#DDDDDD', zorder=0)
ax.grid(axis='y', which='minor', linewidth=0.2, color='#F5F5F5', zorder=0)
ax.set_axisbelow(True)

# Clean spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# ── Legend ────────────────────────────────────
legend_elements = [
    Patch(facecolor=palette[-1], edgecolor='#8A847A', hatch='//', alpha=0.9,
          label='Published KalshiBench best'),
    Patch(facecolor=palette[0], edgecolor='white', label='Single models'),
    Patch(facecolor=palette[1], edgecolor='white', label='Independent aggregation'),
    Patch(facecolor=palette[2], edgecolor='white', label='Deliberative consensus'),
]

ax.legend(
    handles=legend_elements,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.11),
    ncol=2,
    frameon=False,
    handlelength=1.2,
    handleheight=1.0,
    columnspacing=1.6
)

plt.tight_layout(rect=[0, 0.02, 1, 1])

# ── Save ────────────────────────────────────────
fig.savefig(output_dir / 'figure1_overall_accuracy.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(output_dir / 'figure1_overall_accuracy.pdf', bbox_inches='tight', facecolor='white')

plt.show()


## Figure 2
Coverage-Accuracy Tradeoff for Escalation Policy

In [ ]:
comparison_path = Path('../results/escalation_analysis/coverage_accuracy_comparison.csv')
if not comparison_path.exists():
    comparison_path = Path('results/escalation_analysis/coverage_accuracy_comparison.csv')
comparison = pd.read_csv(comparison_path)

fig, ax = plt.subplots(figsize=(8.8, 5.6), facecolor='white')

arch_a_color = '#3A8A5C'
arch_b_color = '#C26A4A'

ax.plot(
    comparison['fraction_auto_resolved'],
    comparison['architecture_a_accuracy'],
    linewidth=2.4,
    color=arch_a_color,
    label='Architecture A',
)
ax.plot(
    comparison['fraction_auto_resolved'],
    comparison['architecture_b_accuracy'],
    linewidth=2.4,
    color=arch_b_color,
    label='Architecture B',
)

annotation_points = [
    {
        'x': 0.474,
        'y': 0.9787,
        'label': 'Unanimous + High Conf',
        'xytext': (12, -58),
    },
    {
        'x': 0.823,
        'y': 0.8834,
        'label': 'All Unanimous',
        'xytext': (28, 30),
    },
]

for point in annotation_points:
    ax.scatter(
        point['x'],
        point['y'],
        s=64,
        color=arch_a_color,
        edgecolor='white',
        linewidth=0.9,
        zorder=5,
    )
    ax.annotate(
        point['label'],
        xy=(point['x'], point['y']),
        xytext=point['xytext'],
        textcoords='offset points',
        fontsize=8.5,
        color='#444444',
        ha='left',
        va='center',
        arrowprops={'arrowstyle': '-', 'color': '#777777', 'lw': 0.8},
    )

ax.set_xlim(0, 1)
ax.set_ylim(0.70, 1.02)
ax.set_xlabel('Coverage (Fraction Auto-Resolved)')
ax.set_ylabel('Accuracy on Auto-Resolved Set')

ax.yaxis.set_major_locator(mticker.FixedLocator([0.75, 0.80, 0.85, 0.90, 0.95, 1.00]))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.grid(axis='both', color='#DDDDDD', linewidth=0.6)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_facecolor('white')
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(output_dir / 'figure2_coverage_accuracy_tradeoff.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(output_dir / 'figure2_coverage_accuracy_tradeoff.pdf', bbox_inches='tight', facecolor='white')

plt.show()


## Figure 3
Per-Model Decision Revisions During Deliberation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle, Polygon, Patch

figure3_data = {
    'GPT-4o': {
        'stayed_correct': 868,
        'correct_to_incorrect': 26,
        'incorrect_to_correct': 16,
        'stayed_incorrect': 279,
        'round1_correct': 894,
        'round1_incorrect': 295,
        'round2_correct': 884,
        'round2_incorrect': 305,
        'net': -10,
    },
    'DeepSeek': {
        'stayed_correct': 903,
        'correct_to_incorrect': 3,
        'incorrect_to_correct': 9,
        'stayed_incorrect': 274,
        'round1_correct': 906,
        'round1_incorrect': 283,
        'round2_correct': 912,
        'round2_incorrect': 277,
        'net': 6,
    },
    'Llama': {
        'stayed_correct': 890,
        'correct_to_incorrect': 16,
        'incorrect_to_correct': 18,
        'stayed_incorrect': 265,
        'round1_correct': 906,
        'round1_incorrect': 283,
        'round2_correct': 908,
        'round2_incorrect': 281,
        'net': 2,
    },
}

total_pairs = 1189
colors = {
    'stayed_correct': '#3A8A5C',
    'stayed_incorrect': '#C26A4A',
    'correct_to_incorrect': '#C0392B',
    'incorrect_to_correct': '#2ECC71',
    'block_fill': '#F5F3EF',
    'block_edge': '#CFCFCF',
    'text': '#333333',
}

def draw_flow(ax, x0, x1, y0_top, y0_bottom, y1_top, y1_bottom, color, alpha=0.92, zorder=2):
    t = np.linspace(0, 1, 160)
    s = 3 * t**2 - 2 * t**3
    xs = x0 + (x1 - x0) * t
    top = y0_top + (y1_top - y0_top) * s
    bottom = y0_bottom + (y1_bottom - y0_bottom) * s
    verts = np.column_stack([
        np.concatenate([xs, xs[::-1]]),
        np.concatenate([top, bottom[::-1]])
    ])
    ax.add_patch(Polygon(verts, closed=True, facecolor=color, edgecolor='none', alpha=alpha, zorder=zorder))

def add_count_label(ax, x, y, text, color, dy=0.0):
    ax.text(
        x,
        y + dy,
        text,
        ha='center',
        va='center',
        fontsize=8.5,
        color=color,
        fontweight='bold',
        bbox={
            'boxstyle': 'round,pad=0.18',
            'facecolor': 'white',
            'edgecolor': 'none',
            'alpha': 0.88
        },
        zorder=6,
    )

fig, ax = plt.subplots(figsize=(11.6, 6.9), facecolor='white')

left_block_x0, left_block_x1 = 0.18, 0.23
right_block_x0, right_block_x1 = 0.77, 0.82
flow_x0, flow_x1 = left_block_x1, right_block_x0

strip_height = 0.20
inner_gap = 0.012

# moved the strips upward to reduce dead space below subtitle
strip_tops = [1.00, 0.72, 0.44]

# move Round labels down so they sit closer to the chart body
round_label_y = strip_tops[0] + 0.008
ax.text((left_block_x0 + left_block_x1) / 2, round_label_y, 'Round 1',
        ha='center', va='bottom', fontsize=10, color=colors['text'])
ax.text((right_block_x0 + right_block_x1) / 2, round_label_y, 'Round 2',
        ha='center', va='bottom', fontsize=10, color=colors['text'])

for strip_top, (model_name, model) in zip(strip_tops, figure3_data.items()):
    strip_bottom = strip_top - strip_height
    available_height = strip_height - inner_gap

    l_correct_h = available_height * model['round1_correct'] / total_pairs
    l_incorrect_h = available_height * model['round1_incorrect'] / total_pairs
    r_correct_h = available_height * model['round2_correct'] / total_pairs
    r_incorrect_h = available_height * model['round2_incorrect'] / total_pairs

    l_correct_top = strip_top
    l_correct_bottom = l_correct_top - l_correct_h
    l_incorrect_bottom = strip_bottom
    l_incorrect_top = l_incorrect_bottom + l_incorrect_h

    r_correct_top = strip_top
    r_correct_bottom = r_correct_top - r_correct_h
    r_incorrect_bottom = strip_bottom
    r_incorrect_top = r_incorrect_bottom + r_incorrect_h

    ax.add_patch(Rectangle(
        (left_block_x0, l_correct_bottom),
        left_block_x1 - left_block_x0,
        l_correct_h,
        facecolor=colors['block_fill'],
        edgecolor=colors['block_edge'],
        linewidth=0.8,
        zorder=1
    ))
    ax.add_patch(Rectangle(
        (left_block_x0, l_incorrect_bottom),
        left_block_x1 - left_block_x0,
        l_incorrect_h,
        facecolor=colors['block_fill'],
        edgecolor=colors['block_edge'],
        linewidth=0.8,
        zorder=1
    ))
    ax.add_patch(Rectangle(
        (right_block_x0, r_correct_bottom),
        right_block_x1 - right_block_x0,
        r_correct_h,
        facecolor=colors['block_fill'],
        edgecolor=colors['block_edge'],
        linewidth=0.8,
        zorder=1
    ))
    ax.add_patch(Rectangle(
        (right_block_x0, r_incorrect_bottom),
        right_block_x1 - right_block_x0,
        r_incorrect_h,
        facecolor=colors['block_fill'],
        edgecolor=colors['block_edge'],
        linewidth=0.8,
        zorder=1
    ))

    sc_h = available_height * model['stayed_correct'] / total_pairs
    ci_h = available_height * model['correct_to_incorrect'] / total_pairs
    ic_h = available_height * model['incorrect_to_correct'] / total_pairs
    si_h = available_height * model['stayed_incorrect'] / total_pairs

    left_sc_top = l_correct_top
    left_sc_bottom = left_sc_top - sc_h
    left_ci_top = left_sc_bottom
    left_ci_bottom = left_ci_top - ci_h

    left_ic_top = l_incorrect_top
    left_ic_bottom = left_ic_top - ic_h
    left_si_top = left_ic_bottom
    left_si_bottom = l_incorrect_bottom

    right_sc_top = r_correct_top
    right_sc_bottom = right_sc_top - sc_h
    right_ic_top = right_sc_bottom
    right_ic_bottom = r_correct_bottom

    right_ci_top = r_incorrect_top
    right_ci_bottom = right_ci_top - ci_h
    right_si_top = right_ci_bottom
    right_si_bottom = r_incorrect_bottom

    draw_flow(ax, flow_x0, flow_x1,
              left_sc_top, left_sc_bottom,
              right_sc_top, right_sc_bottom,
              colors['stayed_correct'], zorder=2)
    draw_flow(ax, flow_x0, flow_x1,
              left_si_top, left_si_bottom,
              right_si_top, right_si_bottom,
              colors['stayed_incorrect'], zorder=2)
    draw_flow(ax, flow_x0, flow_x1,
              left_ci_top, left_ci_bottom,
              right_ci_top, right_ci_bottom,
              colors['correct_to_incorrect'], zorder=4)
    draw_flow(ax, flow_x0, flow_x1,
              left_ic_top, left_ic_bottom,
              right_ic_top, right_ic_bottom,
              colors['incorrect_to_correct'], zorder=4)

    ax.text(0.085, strip_top - strip_height / 2, model_name,
            ha='right', va='center', fontsize=11,
            fontweight='bold', color=colors['text'])

    ax.text(left_block_x0 - 0.016, (l_correct_top + l_correct_bottom) / 2,
            'Correct', ha='right', va='center', fontsize=8.8, color=colors['text'])
    ax.text(left_block_x0 - 0.016, (l_incorrect_top + l_incorrect_bottom) / 2,
            'Incorrect', ha='right', va='center', fontsize=8.8, color=colors['text'])
    ax.text(right_block_x1 + 0.016, (r_correct_top + r_correct_bottom) / 2,
            'Correct', ha='left', va='center', fontsize=8.8, color=colors['text'])
    ax.text(right_block_x1 + 0.016, (r_incorrect_top + r_incorrect_bottom) / 2,
            'Incorrect', ha='left', va='center', fontsize=8.8, color=colors['text'])

    harmful_center_y = ((left_ci_top + left_ci_bottom) / 2 + (right_ci_top + right_ci_bottom) / 2) / 2
    helpful_center_y = ((left_ic_top + left_ic_bottom) / 2 + (right_ic_top + right_ic_bottom) / 2) / 2

    add_count_label(ax, 0.505, harmful_center_y + 0.018,
                    str(model['correct_to_incorrect']),
                    colors['correct_to_incorrect'])
    add_count_label(ax, 0.545, helpful_center_y - 0.018,
                    str(model['incorrect_to_correct']),
                    colors['incorrect_to_correct'])

    net = model['net']
    net_color = colors['correct_to_incorrect'] if net < 0 else colors['stayed_correct']
    ax.text(0.92, strip_top - strip_height / 2, f'Net: {net:+d}',
            ha='left', va='center', fontsize=10,
            fontweight='bold', color=net_color)

legend_handles = [
    Patch(facecolor=colors['stayed_correct'], edgecolor='none', label='Stayed correct'),
    Patch(facecolor=colors['correct_to_incorrect'], edgecolor='none', label='Correct -> Incorrect'),
    Patch(facecolor=colors['incorrect_to_correct'], edgecolor='none', label='Incorrect -> Correct'),
    Patch(facecolor=colors['stayed_incorrect'], edgecolor='none', label='Stayed incorrect'),
]

ax.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.14),
    ncol=4,
    frameon=False,
    fontsize=8.5,
    columnspacing=1.6
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_facecolor('white')

fig.tight_layout(rect=[0.02, 0.01, 0.98, 0.98])

fig.savefig(output_dir / 'figure3_model_revisions_architecture_b.png',
            dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(output_dir / 'figure3_model_revisions_architecture_b.pdf',
            bbox_inches='tight', facecolor='white')

plt.show()


## Figure 4
Diverging Revision Balance by Model

In [ ]:
revision_models = ['GPT-4o', 'DeepSeek', 'Llama']
harmful_flips = np.array([26, 3, 16])
helpful_flips = np.array([16, 9, 18])
net_effect = helpful_flips - harmful_flips

y = np.arange(len(revision_models))
fig, ax = plt.subplots(figsize=(9.4, 4.8), facecolor='white')

harmful_color = '#C0392B'
helpful_color = '#2ECC71'
neutral_color = '#666666'

ax.barh(y, -harmful_flips, color=harmful_color, height=0.56, edgecolor='none', label='Correct -> Incorrect')
ax.barh(y, helpful_flips, color=helpful_color, height=0.56, edgecolor='none', label='Incorrect -> Correct')
ax.axvline(0, color='#BDBDBD', linewidth=1.0)

for yi, harmful, helpful, net in zip(y, harmful_flips, helpful_flips, net_effect):
    ax.text(-harmful - 0.9, yi, str(harmful), ha='right', va='center', fontsize=9, color=harmful_color, fontweight='bold')
    ax.text(helpful + 0.9, yi, str(helpful), ha='left', va='center', fontsize=9, color=helpful_color, fontweight='bold')
    net_color = harmful_color if net < 0 else helpful_color
    ax.text(29.2, yi, f'Net: {net:+d}', ha='left', va='center', fontsize=10, color=net_color, fontweight='bold')

ax.set_yticks(y)
ax.set_yticklabels(revision_models)
ax.invert_yaxis()
ax.set_xlim(-30, 34)
ax.set_xlabel('Revision Count')
ax.text(0.5, 1.02, 'Only revised decisions shown: harmful flips left, helpful flips right.', transform=ax.transAxes, ha='center', va='bottom', fontsize=9, color='#555555')

ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
ax.grid(axis='x', color='#DDDDDD', linewidth=0.6)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_color('#CCCCCC')
ax.tick_params(axis='y', length=0)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False)

fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(output_dir / 'figure4_diverging_revisions_architecture_b.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(output_dir / 'figure4_diverging_revisions_architecture_b.pdf', bbox_inches='tight', facecolor='white')

plt.show()


## Figure 5
Category-Level Resolution Accuracy on KalshiBench

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8.5,
})

output_dir = Path('figures')
output_dir.mkdir(exist_ok=True)

# Data
categories = [
    'Education (n=2)',
    'Health (n=2)',
    'Financials (n=30)',
    'Sports (n=269)',
    'Social (n=13)',
    'Elections (n=80)',
    'Entertainment (n=230)',
    'World (n=29)',
    'Mentions (n=52)',
    'Politics (n=311)',
    'Climate & Weather (n=27)',
    'Economics (n=21)',
    'Crypto (n=59)',
    'Sci & Tech (n=10)',
    'Companies (n=50)',
    'Transportation (n=4)',
]

a_weighted = np.array([100.0, 100.0, 93.3, 92.9, 92.3, 91.3, 87.8, 86.2, 80.8, 79.4, 70.4, 66.7, 66.1, 60.0, 58.0, 50.0])
best_baseline = np.array([100.0, 50.0, 90.0, 92.2, 76.9, 95.0, 87.8, 86.2, 76.9, 76.2, 77.8, 57.1, 67.8, 40.0, 62.0, 75.0])
b_delib = np.array([100.0, 50.0, 86.7, 82.9, 69.2, 85.0, 79.6, 89.7, 75.0, 73.0, 66.7, 42.9, 59.3, 40.0, 68.0, 25.0])

y = np.arange(len(categories))[::-1]

# Colors
green = '#3A8A5C'
blue = '#4878A8'
orange = '#C26A4A'

fig, ax = plt.subplots(figsize=(10.2, 9.2), facecolor='white')

# Subtle tier shading
ax.axhspan(9.5, 15.5, color='#f6fbf7', zorder=0)
ax.axhspan(6.5, 9.5, color='#fafafa', zorder=0)
ax.axhspan(-0.5, 6.5, color='#fcf8f6', zorder=0)

# Range lines
mins = np.minimum.reduce([a_weighted, best_baseline, b_delib])
maxs = np.maximum.reduce([a_weighted, best_baseline, b_delib])
for yi, lo, hi in zip(y, mins, maxs):
    ax.hlines(yi, lo, hi, color='#b8b8b8', linewidth=1.0, alpha=0.45, zorder=1)

# Smart jitter logic
a_plot = a_weighted.copy()
b_plot = best_baseline.copy()
c_plot = b_delib.copy()
jitter = 0.8

for i in range(len(a_weighted)):
    values = [a_weighted[i], best_baseline[i], b_delib[i]]
    if len(set(values)) < 3:
        if a_weighted[i] == best_baseline[i]:
            a_plot[i] += jitter
            b_plot[i] -= jitter
        if a_weighted[i] == b_delib[i]:
            a_plot[i] += jitter
            c_plot[i] -= jitter
        if best_baseline[i] == b_delib[i]:
            b_plot[i] += jitter
            c_plot[i] -= jitter

# Dots
dot_size = 62
ax.scatter(a_plot, y, s=dot_size, color=green, zorder=3)
ax.scatter(b_plot, y, s=dot_size, color=blue, zorder=3)
ax.scatter(c_plot, y, s=dot_size, color=orange, zorder=3)

# Axes
ax.set_yticks(y)
ax.set_yticklabels(categories, fontsize=10)
ax.set_xlim(18, 103)
ax.set_xticks(np.arange(20, 101, 10))
ax.set_xlabel('Accuracy (%)', fontsize=11)


# Grid and style
ax.xaxis.grid(True, linestyle='-', linewidth=0.8, alpha=0.22)
ax.yaxis.grid(False)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Tier divider lines
ax.axhline(9.5, color='#d0d0d0', linewidth=0.8, alpha=0.7)
ax.axhline(6.5, color='#d0d0d0', linewidth=0.8, alpha=0.7)

# Legend
legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=green, markersize=8,
           label='Architecture A (Conf.-Weighted)'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=blue, markersize=8,
           label='Best Single-LLM Baseline'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=orange, markersize=8,
           label='Architecture B (Deliberative)'),
]

leg = ax.legend(
    handles=legend_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.14),
    frameon=True,
    fontsize=9,
    borderpad=0.5,
    handletextpad=0.5,
    labelspacing=0.4,
)

leg.get_frame().set_facecolor('white')
leg.get_frame().set_edgecolor('#dddddd')
leg.get_frame().set_alpha(0.9)

# Layout fix
fig.subplots_adjust(left=0.31, right=0.965, top=0.92, bottom=0.08)

fig.savefig(output_dir / 'figure5_category_dotplot_final.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(output_dir / 'figure5_category_dotplot_final.pdf', bbox_inches='tight', facecolor='white')

plt.show()
